# Predicting malignant findings

This notebook tests whether the anonymized attributes in `data.csv` can predict the binary `class` column (`1` = malignant). Because malignancy is rare, the primary metric is **average precision (AP / area under the precision-recall curve)** rather than accuracy. ROC AUC is included as a secondary ranking metric.

The workflow is deliberately leakage-aware: rows with identical feature values are assigned to the same cross-validation fold. This matters because the dataset contains many exact duplicate feature profiles.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    average_precision_score, classification_report, confusion_matrix,
    precision_recall_curve, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 1. Load and validate the data

In [ ]:
DATA_PATH = Path('data.csv')
df = pd.read_csv(DATA_PATH)

assert 'class' in df.columns, "Expected a target column named 'class'."
assert set(df['class'].dropna().unique()).issubset({0, 1}), 'class must be binary (0/1).'
assert df['class'].notna().all(), 'The target contains missing values.'

feature_cols = [c for c in df.columns if c != 'class']
non_numeric = df[feature_cols].select_dtypes(exclude=np.number).columns.tolist()
assert not non_numeric, f'Non-numeric features require preprocessing: {non_numeric}'

summary = pd.DataFrame({
    'rows': [len(df)],
    'features': [len(feature_cols)],
    'missing feature values': [int(df[feature_cols].isna().sum().sum())],
    'malignant': [int(df['class'].sum())],
    'malignant rate': [df['class'].mean()],
})
display(summary)
display(df.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df['class'].value_counts().sort_index()
axes[0].bar(['benign (0)', 'malignant (1)'], counts.values, color=['#4C78A8', '#E45756'])
axes[0].set_title('Strong target imbalance')
axes[0].set_ylabel('Rows')

mean_by_class = df.groupby('class')[feature_cols].mean().T
mean_by_class.plot(kind='bar', ax=axes[1], color=['#4C78A8', '#E45756'])
axes[1].set_title('Feature means by class')
axes[1].set_ylabel('Mean value')
axes[1].legend(title='class')
plt.tight_layout();

## 2. Duplicate-profile audit

A random row split would allow identical feature vectors to occur in training and validation, inflating performance. We therefore hash the six predictors and use the hash as a grouping variable. Conflicting labels within an identical profile also reveal cases no model using only these features can distinguish.

In [ ]:
X = df[feature_cols].copy()
y = df['class'].astype(int).copy()
groups = pd.util.hash_pandas_object(X, index=False)

profile_table = (df.assign(_profile=groups)
                   .groupby('_profile')
                   .agg(rows=('class', 'size'), malignant=('class', 'sum'), labels=('class', 'nunique'))
                   .sort_values('rows', ascending=False))
duplicate_rows = int(df.duplicated(subset=feature_cols, keep=False).sum())
conflicting_profiles = int((profile_table['labels'] > 1).sum())

print(f'Unique feature profiles: {groups.nunique():,} of {len(df):,} rows')
print(f'Rows belonging to a repeated profile: {duplicate_rows:,}')
print(f'Profiles associated with both labels: {conflicting_profiles:,}')
display(profile_table.head())

largest_profile = profile_table.index[0]
display(df.loc[groups.eq(largest_profile)].groupby(feature_cols, dropna=False)['class']
        .agg(['size', 'sum']).rename(columns={'sum': 'malignant'}))

## 3. Leakage-aware model comparison

We compare a prevalence-only baseline with a balanced logistic regression and two nonlinear tree ensembles. Every row receives an out-of-fold prediction from a model that was not trained on that row **or any identical feature profile**.

Accuracy is intentionally omitted from model selection: predicting every row as benign would already be highly accurate. AP should be compared with the malignant prevalence (the random-ranking baseline).

In [ ]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    'Prevalence baseline': DummyClassifier(strategy='prior'),
    'Balanced logistic regression': make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight='balanced', max_iter=2_000, random_state=RANDOM_STATE),
    ),
    'Balanced random forest': RandomForestClassifier(
        n_estimators=500, min_samples_leaf=2, class_weight='balanced_subsample',
        n_jobs=-1, random_state=RANDOM_STATE,
    ),
    'Balanced histogram boosting': HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.08, l2_regularization=1.0,
        class_weight='balanced', random_state=RANDOM_STATE,
    ),
}

oof_probabilities = {}
rows = []
for name, model in models.items():
    if name == 'Prevalence baseline':
        probabilities = np.full(len(y), y.mean())
    else:
        probabilities = cross_val_predict(
            model, X, y, groups=groups, cv=cv, method='predict_proba', n_jobs=-1
        )[:, 1]
    oof_probabilities[name] = probabilities
    rows.append({
        'model': name,
        'average precision': average_precision_score(y, probabilities),
        'ROC AUC': roc_auc_score(y, probabilities),
        'AP lift vs prevalence': average_precision_score(y, probabilities) / y.mean(),
    })

results = pd.DataFrame(rows).sort_values('average precision', ascending=False).reset_index(drop=True)
display(results.style.format({
    'average precision': '{:.3f}', 'ROC AUC': '{:.3f}', 'AP lift vs prevalence': '{:.1f}×'
}))
print(f'Random-ranking AP baseline (malignant prevalence): {y.mean():.3%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, probabilities in oof_probabilities.items():
    PrecisionRecallDisplay.from_predictions(y, probabilities, name=name, ax=axes[0])
    RocCurveDisplay.from_predictions(y, probabilities, name=name, ax=axes[1])
axes[0].axhline(y.mean(), color='gray', linestyle='--', label='prevalence')
axes[0].set_title('Out-of-fold precision-recall curves')
axes[1].set_title('Out-of-fold ROC curves')
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout();

## 4. Operating threshold

A ranking model does not determine a clinical cutoff. As an illustrative screening-oriented rule, we choose the threshold that maximizes the F2 score on the out-of-fold predictions, weighting recall twice as heavily as precision. This is exploratory; the final cutoff should reflect the actual cost of a missed malignancy versus a false alarm and be validated prospectively.

In [ ]:
best_name = results.loc[0, 'model']
best_prob = oof_probabilities[best_name]
precision, recall, thresholds = precision_recall_curve(y, best_prob)
f2 = 5 * precision * recall / (4 * precision + recall + 1e-12)
best_index = int(np.nanargmax(f2[:-1]))
threshold = float(thresholds[best_index])
predicted = (best_prob >= threshold).astype(int)

print(f'Selected model: {best_name}')
print(f'Illustrative F2-optimal threshold: {threshold:.3f}')
print(classification_report(y, predicted, target_names=['benign', 'malignant'], digits=3))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y, predicted),
                       display_labels=['benign', 'malignant']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('OOF confusion matrix at illustrative threshold')
plt.tight_layout();

## 5. Exploratory feature importance

The following model is fitted on all rows to create a final estimator. Training-set permutation importance is shown only as an exploratory description of what that fitted model uses; it is not an unbiased performance estimate or a causal effect, and correlated predictors can divide importance between themselves.

In [ ]:
final_model = clone(models[best_name]).fit(X, y)
permutation = permutation_importance(
    final_model, X, y, scoring='average_precision', n_repeats=10,
    random_state=RANDOM_STATE, n_jobs=-1,
)
importance = pd.Series(permutation.importances_mean, index=feature_cols)
importance.sort_values().plot.barh(figsize=(7, 4), color='#4C78A8')
plt.title(f'Exploratory permutation importance: {best_name}')
plt.xlabel('Mean decrease in training AP after permutation')
plt.tight_layout()
display(importance.sort_values(ascending=False).rename('importance').to_frame())

## 6. Conclusion and limitations

The result table above answers the modeling question. A useful signal exists if the best model's out-of-fold AP is materially above the malignant prevalence and its ROC AUC is above 0.5. The grouped evaluation is stricter than a random row split because identical profiles cannot leak across folds.

Important limitations:

- This is an internal, exploratory validation on anonymized data—not evidence of clinical safety or generalization to another site, device, or patient population.
- One very large repeated profile contains both labels. No model using only these six attributes can distinguish malignant from benign rows within that profile. This should be investigated as possible missing-value coding, preprocessing collapse, or duplicated records.
- Class imbalance makes accuracy uninformative. Review precision, recall, AP, and the consequences of false negatives and false positives.
- Model and threshold selection use the same grouped out-of-fold predictions. Before deployment, lock both choices and evaluate once on a genuinely external or later-in-time test set.
- Calibration, confidence intervals, subgroup performance, and clinical utility should be assessed once patient/site/time identifiers and feature definitions are available.